In [1]:
from openadmet.toolkit.database.chembl import MicrosomalChEMBLScaler, MicrosomalChEMBLCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm
import subprocess
import os

/Users/cynthiaxu/miniforge3/envs/openadmet-toolkit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Curating basic Microsomal clearance data and pushing to a remote intake catalog

Our goal is to curate activity data from ChEMBL and push this to a remote location with a catalog that can be used by others to look up our data. This will enable consistency and rapid dissemination of our work as well as an over-time evolution of our data curation practices. 

We use the `Intake` package for a lightweight self-describing data parsing workflow. Read more about intake here: https://intake.readthedocs.io/en/latest/index.html


We are gathering microsomal clearance data from ChEMBL accross 3 species, rats, humans and mice. This is important as there is often clearance differences between these three species. 

Here we gather  measurements on the same compound by taking the mean and median. This is the most basic form of curation available, but serves as a good baseline for our initial models. 




## gather ChEMBL data

First we need to gather in our data from ChEMBL using our SQL API defined in `openadmet-toolkit`

We use `OPENADMET_CANONICAL_SMILES` and `OPENADMET_INCHIKEY` to distinguish our ML ready representation from the source SMILES

In [2]:
def gather_microsome_chembl_data_for_species_SCALED(target_name: str, organism: str, chembl_ver: int):
    print(f"working on target {target_name}")
    pctc = MicrosomalChEMBLScaler(organism=organism, version=chembl_ver, standard_type="CL", require_units=["mL.min-1.g-1", "mL.min-1.kg-1"])
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

def gather_microsome_chembl_data_for_species(target_name: str, organism: str, chembl_ver: int):
    print(f"working on target {target_name}")
    pctc = MicrosomalChEMBLCurator(organism=organism, version=chembl_ver, standard_type="CL", require_units=["mL.min-1.g-1", "mL.min-1.kg-1"])
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data
        

In [3]:
def process_and_upload(
    gather_fn,
    target,
    organism,
    chembl_ver,
    location_path,
    location,
    bucket,
    fname_prefix,
):
    agg, raw = gather_fn(target, organism, chembl_ver)

    fname_agg = f"{fname_prefix}_aggregated.parquet"
    fname_raw = f"{fname_prefix}_raw.parquet"

    agg_path = location_path / fname_agg
    raw_path = location_path / fname_raw

    agg.to_parquet(agg_path)
    raw.to_parquet(raw_path)

    dest_agg = f"{location}/{fname_agg}"
    dest_raw = f"{location}/{fname_raw}"

    bucket.push_file(agg_path, dest_agg)
    bucket.push_file(raw_path, dest_raw)

    return (
        bucket.to_uri(dest_agg),
        bucket.to_uri(dest_raw),
    )


# Scale all microsomal data to in vivo CLint

Which means converting all in vitro CLint from units of `ml/min/g` to `ml/min/kg`. This is not a simple matter of converting `g` to `kg`. It is scaled based on the body weight of the organism. See this [reference]("Microsomal_Stability_Study_Report(h,m)_ADME121925c-pages.pdf") for full details on the equation used and the values scaling.

## Define target metadata

We need the CHEMBL codes for our targets

In [4]:
species = {
    "HLM_CL": "Homo sapiens",
    "RLM_CL": "Rattus norvegicus",
    "MLM_CL": "Mus musculus",

}

In [5]:
chembl_ver = 35

In [6]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

# Setup S3

After curating our data we would like to push to a remote bucket to save both the raw data and the catalog

In [7]:
os.environ["AWS_ACCESS_KEY_ID"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_access_key_id"],
    text=True
).strip()

os.environ["AWS_SECRET_ACCESS_KEY"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_secret_access_key"],
    text=True
).strip()

In [8]:
settings = S3Settings()

In [9]:
bucket = "openadmet-data-public-dev"

In [10]:
bucket = S3Bucket.from_settings(settings, bucket)

In [11]:
import datetime

In [12]:
t = datetime.datetime.now()

In [13]:
date = t.strftime("%Y-%m-%d")

In [14]:
location=f"ChEMBL{chembl_ver}_Microsome"

In [15]:
import os
from pathlib import Path

location_path = Path(location)

In [16]:
location_path.mkdir(exist_ok=True)

# Main loop

Generate the data for each target and save to parquet, then push to S3 data lake with parquet files. 

We use parquet here for improved performance and reduced size on disk.

In [17]:
uris_raw_scaled = {}
uris_agg_scaled = {}

uris_raw_unscaled = {}
uris_agg_unscaled = {}

unscaled_path = Path(location)/ "unscaled"
unscaled_path.mkdir(exist_ok=True, parents=True)

scaled_path = Path(location)/ "scaled"
scaled_path.mkdir(exist_ok=True, parents=True)

for target, organism in species.items():

    # UNSCALED
    uri_agg, uri_raw = process_and_upload(
        gather_fn=gather_microsome_chembl_data_for_species,
        target=target,
        organism=organism,
        chembl_ver=chembl_ver,
        location_path=unscaled_path,
        location=location,
        bucket=bucket,
        fname_prefix=f"ChEMBL_{target}_unscaled",
    )

    uris_agg_unscaled[target] = uri_agg
    uris_raw_unscaled[target] = uri_raw

    # SCALED
    uri_agg_scaled, uri_raw_scaled = process_and_upload(
        gather_fn=gather_microsome_chembl_data_for_species_SCALED,
        target=target,
        organism=organism,
        chembl_ver=chembl_ver,
        location_path=scaled_path,
        location=location,
        bucket=bucket,
        fname_prefix=f"ChEMBL_{target}_scaled",
    )

    uris_agg_scaled[target] = uri_agg_scaled
    uris_raw_scaled[target] = uri_raw_scaled


working on target HLM_CL
canonicalising raw data


100%|██████████| 13135/13135 [00:02<00:00, 5152.17it/s]


smiles duplicates 0
inchikey duplicates 0
working on target HLM_CL
canonicalising raw data


100%|██████████| 13135/13135 [00:02<00:00, 5119.58it/s]


smiles duplicates 0
inchikey duplicates 0
working on target RLM_CL
canonicalising raw data


...

smiles duplicates 0
inchikey duplicates 0
working on target MLM_CL
canonicalising raw data


100%|██████████| 4729/4729 [00:00<00:00, 5273.21it/s]


smiles duplicates 0
inchikey duplicates 0
working on target MLM_CL
canonicalising raw data


100%|██████████| 4729/4729 [00:00<00:00, 5298.12it/s]


smiles duplicates 0
inchikey duplicates 0


# Build the Intake Catalog

We have sucessfully aggregted our data and pushed it to a remote destination. Now for others to consume our data, we are going to make an `Intake` catalog such that our data can be readily made available. 

The workflow here is drawn from the `creator` walkthrough from the main intake tutorials https://intake.readthedocs.io/en/latest/walkthrough2.html

TODO: add descriptions to the catalog

In [18]:
import intake

In [19]:
intake.Catalog?

Init signature:
intake.Catalog(
    entries: 'Iterable[ReaderDescription] | Mapping | None' = None,
    aliases: 'dict[str, int] | None' = None,
    data: 'Iterable[DataDescription] | Mapping' = None,
    user_parameters: 'dict[str, BaseUserParameter] | None' = None,
    parameter_overrides: 'dict[str, Any] | None' = None,
    metadata: 'dict | None' = None,
)
Docstring:      A collection of data and reader descriptions.
File:           ~/miniforge3/envs/openadmet-toolkit/lib/python3.12/site-packages/intake/readers/entry.py
Type:           type
Subclasses:     THREDDSCatalog

In [20]:
cat = intake.entry.Catalog()

In [21]:
uris_agg_unscaled

{'HLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_HLM_CL_unscaled_aggregated.parquet',
 'RLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_RLM_CL_unscaled_aggregated.parquet',
 'MLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_MLM_CL_unscaled_aggregated.parquet'}

In [22]:
uris_raw_unscaled

{'HLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_HLM_CL_unscaled_raw.parquet',
 'RLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_RLM_CL_unscaled_raw.parquet',
 'MLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_MLM_CL_unscaled_raw.parquet'}

In [23]:
uris_agg_scaled

{'HLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_HLM_CL_scaled_aggregated.parquet',
 'RLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_RLM_CL_scaled_aggregated.parquet',
 'MLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_MLM_CL_scaled_aggregated.parquet'}

In [24]:
uris_raw_scaled

{'HLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_HLM_CL_scaled_raw.parquet',
 'RLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_RLM_CL_scaled_raw.parquet',
 'MLM_CL': 's3://openadmet-data-public-dev/ChEMBL35_Microsome/ChEMBL_MLM_CL_scaled_raw.parquet'}

In [25]:
for k,v in uris_agg_unscaled.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

for k,v in uris_raw_unscaled.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

for k,v in uris_agg_scaled.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

for k,v in uris_raw_scaled.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

## Push the Catalog

Ok now we have made the catalog, lets push it to the remote location so it can live alongside the data. 

The catalog can then be used from S3 or from github etc, anything that exposes a file-like API. 

In [26]:
cat

Catalog
 named datasets: []

In [27]:
catname = f"CATALOG_{location}.yaml"

In [28]:
cat.to_yaml_file(catname)

In [29]:
cat_location = location+ "/" +catname

In [30]:
cat_location

'ChEMBL35_Microsome/CATALOG_ChEMBL35_Microsome.yaml'

In [31]:
bucket.push_file(catname, cat_location)

In [32]:
cat_uri = bucket.to_uri(cat_location)

In [31]:
# Now can read the catalog from URI
# cat = intake.Catalog.from_yaml_file("s3://openadmet-data-public-dev/ChEMBL34_permissive_2025-02-12/CATALOG_ChEMBL34_permissive_2025-02-12.yaml")